# Models

The `kgcnn_torch` package provides PyTorch implementations of graph neural network models. All models are standard `torch.nn.Module` subclasses that accept PyG `Data` batch objects as input.

In [ ]:
import torch
import numpy as np
from torch_geometric.data import Data, Batch

## Example Data

First, let us create some example graph data in PyG format to demonstrate the models.

In [ ]:
# Create a few small graphs
data1 = Data(
    z=torch.tensor([6, 6, 8]),           # Atomic numbers: C, C, O
    pos=torch.randn(3, 3),               # Random 3D positions
    edge_index=torch.tensor([[0, 1, 1, 2], [1, 0, 2, 1]]),  # (2, M) edges
    edge_weight=torch.ones(4, 1),        # Edge weights for GCN
    y=torch.tensor([1.5]),               # Target property
)

data2 = Data(
    z=torch.tensor([6, 7, 6, 8]),
    pos=torch.randn(4, 3),
    edge_index=torch.tensor([[0, 1, 1, 2, 2, 3], [1, 0, 2, 1, 3, 2]]),
    edge_weight=torch.ones(6, 1),
    y=torch.tensor([2.3]),
)

# Batch them together (as PyG DataLoader would)
batch = Batch.from_data_list([data1, data2])
print("Batch:", batch)
print("  z:", batch.z)
print("  batch:", batch.batch)
print("  edge_index shape:", batch.edge_index.shape)

## GCN Model

The `GCNModel` implements the Graph Convolutional Network (Kipf & Welling, 2017). It expects integer node features (atomic numbers) by default and uses an embedding layer.

**Expected PyG Data attributes:**
- `data.z` or `data.x`: Node features (integer atomic numbers or float features)
- `data.edge_index`: Edge indices `(2, M)`
- `data.edge_weight`: Edge weights `(M, 1)` (optional, defaults to 1)
- `data.batch`: Batch assignment `(N,)`

In [ ]:
from kgcnn_torch.models.gcn import GCNModel

model_gcn = GCNModel(
    node_dim=64,           # Embedding dimension
    depth=3,               # Number of GCN layers
    gcn_units=100,         # Hidden dimension in GCN layers
    gcn_activation="relu",
    gcn_pooling="sum",
    node_pooling="sum",    # Graph-level readout pooling
    output_units=[25, 10], # Output MLP hidden units
    num_targets=1,         # Number of output targets
    use_node_embedding=True,
    num_embeddings=95,     # Vocabulary size for atomic numbers
)

# Forward pass
with torch.no_grad():
    output = model_gcn(batch)

print("GCN output shape:", output.shape)  # (batch_size, num_targets)
print("GCN predictions:", output)

## SchNet Model

The `SchNetModel` (Schutt et al., 2017) is designed for molecular property prediction using continuous-filter convolutions on atomic positions. It computes inter-atomic distances from `pos` and expands them with Gaussian basis functions.

**Expected PyG Data attributes:**
- `data.z`: Atomic numbers `(N,)`
- `data.pos`: Atom positions `(N, 3)`
- `data.edge_index`: Edge indices `(2, M)`
- `data.batch`: Batch assignment `(N,)`

In [ ]:
from kgcnn_torch.models.schnet import SchNetModel

model_schnet = SchNetModel(
    node_dim=64,               # Embedding dimension
    depth=4,                   # Number of interaction blocks
    units=128,                 # Hidden dimension
    gauss_bins=20,             # Number of Gaussian basis functions
    gauss_distance=4.0,        # Maximum distance for Gaussian expansion
    gauss_sigma=0.4,           # Width of Gaussian basis
    interaction_activation="shifted_softplus",
    interaction_pooling="sum",
    node_pooling="sum",
    last_mlp_units=[128, 64],  # Per-node MLP after interactions
    num_targets=1,
    make_distance=True,        # Compute distances from positions
    expand_distance=True,      # Expand with Gaussian basis
)

with torch.no_grad():
    output = model_schnet(batch)

print("SchNet output shape:", output.shape)
print("SchNet predictions:", output)

## PAiNN Model

The `PAiNNModel` (Schutt et al., 2021) is an equivariant message passing network that maintains both scalar and vector features. It uses Bessel radial basis functions and cosine cutoff envelopes.

**Expected PyG Data attributes:** Same as SchNet.

In [ ]:
from kgcnn_torch.models.painn import PAiNNModel

model_painn = PAiNNModel(
    node_dim=128,
    depth=3,
    units=128,
    num_radial=20,
    cutoff=5.0,
    conv_activation="swish",
    update_activation="swish",
    node_pooling="sum",
    output_units=[128],
    num_targets=1,
)

with torch.no_grad():
    output = model_painn(batch)

print("PAiNN output shape:", output.shape)
print("PAiNN predictions:", output)

## EnergyForceModel

The `EnergyForceModel` wraps any energy-predicting GNN (like SchNet or PAiNN) to also predict forces via automatic differentiation:

$$\vec{F}_i = -\nabla_i E_{\text{total}}$$

The coordinate tensor must have `requires_grad=True` for gradient computation.

In [ ]:
from kgcnn_torch.models.force import EnergyForceModel

# Wrap SchNet to predict both energy and forces
model_ef = EnergyForceModel(
    energy_model=model_schnet,
    coordinate_input="pos",        # Attribute name for coordinates
    output_as_dict=True,           # Return dict with 'energy' and 'force' keys
    output_squeeze_states=True,
    is_physical_force=True,        # F = -grad(E)
)

# Ensure positions require gradient
batch_ef = Batch.from_data_list([data1, data2])
batch_ef.pos.requires_grad_(True)

# Forward pass
result = model_ef(batch_ef)
print("Energy shape:", result['energy'].shape)
print("Force shape:", result['force'].shape)
print("Energy:", result['energy'].detach())
print("Forces (first 3 atoms):\n", result['force'][:3].detach())

## Model Configuration

All models in `kgcnn_torch` are configured via constructor arguments. Key parameters common to most models:

| Parameter | Description |
|---|---|
| `node_dim` | Embedding dimension for node features |
| `depth` | Number of message passing / interaction layers |
| `units` | Hidden dimension in convolution layers |
| `node_pooling` | Graph-level readout pooling method (`"sum"`, `"mean"`, `"max"`) |
| `output_units` | Hidden dimensions for output MLP |
| `num_targets` | Number of output target values |
| `output_embedding` | `"graph"` for graph-level or `"node"` for node-level prediction |
| `use_node_embedding` | Whether to embed integer node features (atomic numbers) |
| `num_embeddings` | Vocabulary size for node embedding (95 covers periodic table) |

## Custom Model from kgcnn_torch Layers

You can build custom GNN models using the layers from `kgcnn_torch.layers`. Here is a simple message passing model.

In [ ]:
import torch.nn as nn
from kgcnn_torch.layers.gather import gather_nodes_outgoing
from kgcnn_torch.layers.aggr import AggregateLocalEdges
from kgcnn_torch.layers.pooling import PoolingNodes


class SimpleMessagePassingModel(nn.Module):
    """A minimal message passing GNN built from kgcnn_torch primitives."""

    def __init__(self, node_dim=64, hidden_dim=64, num_targets=1, num_embeddings=95):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings, node_dim)
        self.message_mlp = nn.Sequential(
            nn.Linear(2 * node_dim, hidden_dim),
            nn.ReLU(),
        )
        self.aggr = AggregateLocalEdges(pooling_method="sum")
        self.update_mlp = nn.Linear(node_dim + hidden_dim, node_dim)
        self.pooling = PoolingNodes(pooling_method="sum")
        self.output_mlp = nn.Linear(node_dim, num_targets)

    def forward(self, data):
        z = data.z
        edge_index = data.edge_index
        batch = data.batch

        # Embed atomic numbers
        x = self.embedding(z.long())
        num_nodes = x.size(0)

        # Message: gather source node features and concatenate with target
        x_source = gather_nodes_outgoing(x, edge_index)  # (M, F)
        x_target = x[edge_index[1]]                       # (M, F)
        messages = self.message_mlp(torch.cat([x_target, x_source], dim=-1))

        # Aggregate messages at target nodes
        agg = self.aggr(messages, edge_index, num_nodes)

        # Update node features
        x = self.update_mlp(torch.cat([x, agg], dim=-1))

        # Graph-level pooling
        batch_size = int(batch.max().item()) + 1
        out = self.pooling(x, batch, batch_size)

        # Output
        return self.output_mlp(out)


custom_model = SimpleMessagePassingModel()
with torch.no_grad():
    output = custom_model(batch)
print("Custom model output:", output)

## Training with trainer.fit()

The `kgcnn_torch.training.trainer` module provides a `fit()` function that handles the complete training loop including validation, logging, learning rate scheduling, and early stopping.

In [ ]:
from torch_geometric.loader import DataLoader
from kgcnn_torch.training.trainer import fit

# Create a small synthetic dataset
dataset = []
for _ in range(30):
    n = np.random.randint(3, 8)
    ei = torch.randint(0, n, (2, n * 2))
    dataset.append(Data(
        z=torch.randint(1, 10, (n,)),
        edge_index=ei,
        edge_weight=torch.ones(ei.size(1), 1),
        y=torch.randn(1),
    ))

train_loader = DataLoader(dataset[:20], batch_size=8, shuffle=True)
val_loader = DataLoader(dataset[20:], batch_size=8, shuffle=False)

# Train the GCN model
model = GCNModel(node_dim=32, depth=2, gcn_units=32, output_units=[16], num_targets=1)

history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=torch.optim.Adam(model.parameters(), lr=1e-3),
    loss_fn=torch.nn.MSELoss(),
    epochs=20,
    verbose=1,
)

print("\nFinal train loss:", history['train_loss'][-1])
print("Final val loss:", history['val_loss'][-1])

> **NOTE**: You can find this page as a Jupyter notebook in the `docs/source` directory of the kgcnn-torch repository.